# RAG Multi Modal Agentique

Ce notebook prépare les bases d’un RAG multimodal : embeddings CLIP, extraction du contenu d’un PDF et préparation de son index de texte dans Chroma. Exécuter les cellules dans l’ordre permet de réutiliser les variables définies aux étapes précédentes.

Les dépendances sont déclarées dans `pyproject.toml`. Les exemples nécessitent `architecture_transformer.webp` et `pdfs/attention is all you need.pdf` dans le dossier de travail. Le chargement initial des poids CLIP peut nécessiter un accès réseau.

La dernière étape d’indexation comporte encore des fonctions auxiliaires à définir, détaillées dans son explication. Le notebook ne contient pas encore de boucle de recherche ni de génération de réponses.


## 01. Importer les bibliothèques multimodales

PyTorch (`torch`) fournit les tenseurs et les calculs sur CPU ou GPU. `open_clip` charge un modèle CLIP pour encoder textes et images dans un espace commun. `PIL.Image` permet de lire les images avant leur prétraitement.


In [2]:
import torch
import open_clip
from PIL import Image

## 02. Choisir le périphérique de calcul

La variable `DEVICE` vaut `cuda` si PyTorch détecte un GPU NVIDIA compatible ; sinon, elle vaut `cpu`. Les entrées et le modèle devront être placés sur ce même périphérique.


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 03. Vérifier le périphérique sélectionné

Cet affichage confirme le périphérique retenu par la cellule précédente. Une sortie `cpu` indique que les calculs seront effectués sur le processeur.


In [5]:
print(f"Using device: {DEVICE}")

Using device: cpu


## 04. Définir le modèle CLIP et ses poids

`MODEL_NAME` choisit l’architecture `ViT-B-32`, tandis que `PRETRAINED` identifie les poids préentraînés à charger. Ces paramètres servent à construire le modèle et à sélectionner son tokeniseur.


In [3]:
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

## 05. Charger le modèle et le prétraitement des images

`create_model_and_transforms` fournit le modèle et les transformations nécessaires à ses entrées visuelles. Les poids peuvent être téléchargés au premier lancement. Le modèle est ensuite déplacé vers `DEVICE` et passé en mode évaluation avec `.eval()`.


In [6]:
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED
)

model = model.to(DEVICE).eval()

## 06. Préparer le tokeniseur de texte

Le tokeniseur associé à `MODEL_NAME` convertit les chaînes de caractères en séquences de jetons au format attendu par l’encodeur de texte CLIP.


In [7]:
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

## 07. Tokeniser un texte d’exemple

La phrase d’exemple est placée dans une liste afin de constituer un lot contenant un seul texte. Le tenseur de jetons est ensuite envoyé sur le même périphérique que le modèle.


In [8]:
texte = "Un chat sur un canapé"
tokens = tokenizer([texte]).to(DEVICE)

## 08. Prétraiter une image d’exemple

Le fichier `architecture_transformer.webp` doit être présent dans le dossier de travail. L’image est ouverte en RGB puis transformée avec `preprocess`. `unsqueeze(0)` ajoute la dimension du lot avant le transfert vers `DEVICE`.


In [10]:
with Image.open("architecture_transformer.webp") as image:
    tenseur_image = preprocess(image.convert("RGB"))

tenseur_image = tenseur_image.unsqueeze(0).to(DEVICE)